# Live League Games

In [1]:
%load_ext autoreload
%autoreload 2
import requests
from dotenv import load_dotenv
import os
import pandas as pd
from datetime import datetime as dt
from sqlalchemy.engine import create_engine, URL
import numpy as np
import json
from retry import retry 
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)
from src.config import ROOT_DIR

In [2]:
# Steam API constants
STEAM_URL = 'http://api.steampowered.com/'
LIVE_LEAGUE_GAMES = 'IDOTA2Match_570/GetLiveLeagueGames/v1'
REAL_TIME_STATS = 'IDOTA2MatchStats_570/GetRealtimeStats/v1' # Requires server_steam_id 

load_dotenv()
API_KEY = os.getenv('STEAM_API')

### Fetching premium and professional leagues' match details

In [3]:
# Function to retrieve Json from Steam WebAPI
session = requests.Session()
session.params.update({'key': API_KEY})

@retry(tries=3, delay=2)
def fetch_live_league_games():
    try:
        url = f'{STEAM_URL}{LIVE_LEAGUE_GAMES}'
        res = session.get(url)
        match_details = res.json()
        if not match_details:
            raise ValueError("Empty dictionary, retrying...")
        else:
            return match_details
    except Exception as err:
        print("Did not get a response, retrying...")
        raise

In [4]:
game_data = fetch_live_league_games()
games = game_data['result']['games']

In [5]:
games_df = pd.DataFrame(games)
games_df

,players,radiant_team,dire_team,lobby_id,match_id,spectators,league_id,league_node_id,stream_delay_s,radiant_series_wins,dire_series_wins,series_type,scoreboard
0,"[{'account_id': 1514212987, 'name': 'Madness',...","{'team_name': 'Immortal Squad', 'team_id': 933...","{'team_name': 'Hellspawn', 'team_id': 8969893,...",29273292536183906,8259132278,0,17911,0,120,1,0,1,"{'duration': 0, 'roshan_respawn_timer': 0, 'ra..."
1,"[{'account_id': 1876781090, 'name': 'FACEIT.co...",NaN,NaN,29273292532978875,8259128412,0,17599,0,900,0,0,0,NaN
2,"[{'account_id': 1609832974, 'name': 'FACEIT.co...",NaN,NaN,29273292530072744,8259125974,0,17599,0,900,0,0,0,NaN
3,"[{'account_id': 297897091, 'name': 'ичпучмак в...","{'team_name': 'Bright Crusaders', 'team_id': 8...","{'team_name': 'Dark Rebellion', 'team_id': 942...",29273292517679996,8259114825,6,17233,35,120,1,0,1,"{'duration': 839.5333862304688, 'roshan_respaw..."
4,"[{'account_id': 147603864, 'name': 'ENenEM', '...","{'team_name': 'Mister Maniacs', 'team_id': 933...","{'team_name': 'Dominatrix', 'team_id': 9332046...",29273292506551867,8259101343,6,16325,251,120,0,0,1,"{'duration': 1714.7001953125, 'roshan_respawn_..."
5,"[{'account_id': 1609981281, 'name': 'VSCL Robo...","{'team_name': 'СИБГУТИ (З)', 'team_id': 955498...","{'team_name': 'НГПУ (Г)', 'team_id': 9564390, ...",29273292534287695,8259129765,1,11845,0,300,0,0,0,"{'duration': 0, 'roshan_respawn_timer': 0, 'ra..."
6,"[{'account_id': 1101870360, 'name': 'Robot VSC...","{'team_name': 'СИУ РАНХиГС (Б)', 'team_id': 92...","{'team_name': 'НГТУ (П)', 'team_id': 9238547, ...",29273292533383000,8259129237,0,11845,0,300,0,0,0,"{'duration': 0, 'roshan_respawn_timer': 0, 'ra..."
7,"[{'account_id': 1056003447, 'name': 'Mr Yolo',...",NaN,NaN,29273292516569353,8259112926,0,17531,0,120,0,0,0,"{'duration': 797.2000122070312, 'roshan_respaw..."


In [16]:
match = games_df.head(1)

In [14]:
json.loads(match.to_json(orient='records', date_format='iso'))

[{'players': [{'account_id': 1610102240,
    'name': 'VSCL Robot',
    'hero_id': 0,
    'team': 4},
   {'account_id': 227466189, 'name': 'Korablik', 'hero_id': 34, 'team': 0},
   {'account_id': 161921318, 'name': 'Slavtys', 'hero_id': 37, 'team': 0},
   {'account_id': 367386018, 'name': 'dlki', 'hero_id': 64, 'team': 0},
   {'account_id': 177420132, 'name': 'ilyamusk', 'hero_id': 29, 'team': 0},
   {'account_id': 162990064, 'name': 'sleeper', 'hero_id': 5, 'team': 1},
   {'account_id': 1210814813,
    'name': 'neverever/^?',
    'hero_id': 128,
    'team': 1},
   {'account_id': 1111442863, 'name': 'prince', 'hero_id': 46, 'team': 1},
   {'account_id': 355489974, 'name': 'THE_KoT', 'hero_id': 44, 'team': 0},
   {'account_id': 943011972, 'name': 'Sonyxy', 'hero_id': 7, 'team': 1},
   {'account_id': 897850169, 'name': 'Rominator', 'hero_id': 113, 'team': 1}],
  'radiant_team': {'team_name': 'НГТУ (П)',
   'team_id': 9238547,
   'team_logo': 2465249148807388740,
   'complete': True},
  'd

In [6]:
# Import the list of premium and professional league games id

import yaml

file_path = os.path.join(ROOT_DIR, f'constants/league_ids.yml')

with open(file_path, 'r') as file:
    content = yaml.safe_load(file) or {}
    if 'PREMIUM_LEAGUES' in content:
        premium_leagues = content['PREMIUM_LEAGUES']
    if 'PROFESSIONAL_LEAGUES' in content:
        professional_leagues = content['PROFESSIONAL_LEAGUES']
        
premium_list = list(premium_leagues.values())
professional_list = list(professional_leagues.values())



In [7]:
from src.pydantic_models.match import Match
from src.pydantic_models.live_league_games import LiveLeagueGames

In [8]:
live_league_games = []

for row in games:
    
    league_id = row.get('league_id', np.nan)
    if league_id in premium_list + professional_list:
    
        game_data = LiveLeagueGames(**row)
        
        # Populate common fields
        match_data = {
            'match_id': game_data.match_id,
            'radiant_team_id': game_data.radiant_team.team_id,
            'radiant_name': game_data.radiant_team.team_name,
            'dire_team_id': game_data.dire_team.team_id,
            'dire_name': game_data.dire_team.team_name,
            'duration': game_data.scoreboard.duration,
            'start_time': int(dt.now().timestamp())
        }
        
        # Populate player data
        for team in ['radiant', 'dire']:
            faction = getattr(game_data.scoreboard, team)
            for player in faction.players:
                slot = player.player_slot
                player_data = {
                    f"slot_{slot}_account_id": player.account_id,
                    f"slot_{slot}_hero_id": player.hero_id
                } 
                match_data.update(player_data)
                
        live_league_games.append(Match(**match_data))
                

    
if len(live_league_games) == 0:
    print("No premium or professional games right now")
else:
    print(len(live_league_games))
    print(live_league_games)   


No premium or professional games right now


In [20]:
live_league_games

[Match(match_id=8259083479, radiant_name='Romashku', radiant_team_id=9735994, dire_name='Nethercore', dire_team_id=9593609, start_time=1744966361, duration=585.5668334960938, radiant_win=None, slot_0_hero_id=5, slot_1_hero_id=79, slot_2_hero_id=16, slot_3_hero_id=49, slot_4_hero_id=106, slot_128_hero_id=46, slot_129_hero_id=108, slot_130_hero_id=100, slot_131_hero_id=26, slot_132_hero_id=126, slot_0_account_id=337792133, slot_1_account_id=1674492970, slot_2_account_id=1885001216, slot_3_account_id=1885118437, slot_4_account_id=1683752803, slot_128_account_id=1029972951, slot_129_account_id=314272971, slot_130_account_id=1712916118, slot_131_account_id=1695617010, slot_132_account_id=1810099486)]